# Assignment A3: Embeddings and Parsing

Covering material from notebooks 7 and 8 

# Word Embeddings

**Training word2vec**

In this section, we train a word2vec model using gensim. We train the model on text8 (which consists of the first 90M characters of a Wikipedia dump from 2006 and is considered one of the benchmarks for evaluating language models).

In [1]:
import gensim.downloader as api

api.info("text8")

{'num_records': 1701,
 'record_format': 'list of str (tokens)',
 'file_size': 33182058,
 'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/text8/__init__.py',
 'license': 'not found',
 'description': 'First 100,000,000 bytes of plain text from Wikipedia. Used for testing purposes; see wiki-english-* for proper full Wikipedia datasets.',
 'checksum': '68799af40b6bda07dfa47a32612e5364',
 'file_name': 'text8.gz',
 'read_more': ['http://mattmahoney.net/dc/textdata.html'],
 'parts': 1}

In [2]:
dataset = api.load("text8")

In [3]:
from gensim.models import Word2Vec

##TODO train a word2vec model on this dataset, only consider words which appear at least 10 times in the corpus

# Train a Word2Vec model on the text8 corpus.
# - `dataset` is an iterable of tokenized sentences (lists of words), which is
#   exactly the format Word2Vec expects as its `sentences` argument.
# - `min_count=10` tells gensim to ignore any word that appears fewer than
#   10 times in the corpus, building the vocabulary only from frequent words.
# - `workers=4` uses multiple CPU threads to speed up training.
model = Word2Vec(sentences=dataset, min_count=10, workers=4)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


**Word Similarities**

gensim models provide almost all the utility you might want to wish for to perform standard word similarity tasks. They are available in the .wv (wordvectors) attribute of the model, more details could be found [here](https://radimrehurek.com/gensim/models/keyedvectors.html).

In [4]:
model.wv

##TODO find the closest words to king

# `most_similar` returns the words whose vectors have the highest cosine
# similarity to the query word "king", ranked from most to least similar.
# Each result is a (word, similarity_score) tuple; topn controls how many we get.
model.wv.most_similar("king", topn=10)

[('prince', 0.7486765384674072),
 ('emperor', 0.7302443385124207),
 ('throne', 0.7039909958839417),
 ('queen', 0.6984345316886902),
 ('kings', 0.6977327466011047),
 ('vii', 0.6894799470901489),
 ('regent', 0.6871732473373413),
 ('aragon', 0.6680324077606201),
 ('constantine', 0.6662178635597229),
 ('charlemagne', 0.6642293930053711)]

King is to man as woman is to X

In [5]:
##TODO find the closest word for the vector "woman" + "king" - "man"

# most_similar can do vector arithmetic directly: words in `positive` are added
# and words in `negative` are subtracted, then it returns the closest words to
# the resulting vector. This computes "woman" + "king" - "man", the classic
# analogy that should yield "queen".
model.wv.most_similar(positive=["woman", "king"], negative=["man"], topn=10)

[('queen', 0.678424060344696),
 ('empress', 0.6443346738815308),
 ('prince', 0.6308053731918335),
 ('princess', 0.6258084774017334),
 ('emperor', 0.6143160462379456),
 ('regent', 0.6055265069007874),
 ('throne', 0.6031582355499268),
 ('son', 0.6030825972557068),
 ('isabella', 0.5980193614959717),
 ('elizabeth', 0.5935655236244202)]

**Evaluate Word Similarities** 

One common way to evaluate word2vec models are word analogy tasks. Let's check how good our model is on one of those. We consider the [WordSim353](http://alfonseca.org/eng/research/wordsim353.html) benchmark, the task is to determine how similar two words are.

In [9]:
# The original host (alfonseca.org) is dead and now redirects to an unrelated site,
# so we fetch the same WordSim353 similarity gold standard from a stable GitHub mirror.
# The file is tab-separated: word1 <tab> word2 <tab> human_similarity_score (203 pairs).
!curl -sL -o wordsim_similarity_goldstandard.txt https://raw.githubusercontent.com/mfaruqui/eval-word-vectors/master/data/word-sim/EN-WS-353-SIM.txt

path = "wordsim_similarity_goldstandard.txt"

def load_data(path):
    X, y = [], []
    with open(path) as f:
        for line in f:
            line = line.strip().split("\t")
            X.append((line[0], line[1])) # each entry in x contains two words, e.g. X[0] = (tiger, cat)
            y.append(float(line[-1])) # each entry in y is the annotation how similar two words are, e.g. Y[0] = 7.35
    return X, y

X, y = load_data(path)
print (X[:3], y[:3])

[('tiger', 'cat'), ('tiger', 'tiger'), ('plane', 'car')] [7.35, 10.0, 5.77]


In [10]:
##TODO compute how similar the pairs in the WordSim353 are according to our model
##TODO if  aword is not present in our model, we assign similarity 0 for the respective text pair

# For each word pair, ask the model how similar the two words are.
# model.wv.similarity(w1, w2) returns the cosine similarity of their vectors,
# but it only works if BOTH words are in the model's vocabulary. Words seen
# fewer than 10 times (our min_count) were dropped, so we guard with `in`
# and fall back to a similarity of 0 when either word is missing.
predictions = []
for w1, w2 in X:
    if w1 in model.wv and w2 in model.wv:
        predictions.append(model.wv.similarity(w1, w2))
    else:
        predictions.append(0)

print(predictions[:3])

[np.float32(0.6377249), np.float32(0.99999994), np.float32(0.44091502)]


In [11]:
from scipy.stats import spearmanr

##TODO compute spearman's rank correlation between our prediction and the human annotations

# Spearman's rank correlation measures how well our model's similarity scores
# (predictions) agree with the human similarity ratings (y) by comparing their
# rankings. spearmanr returns (correlation, p-value); we take [0], the
# correlation, where 1.0 = perfect rank agreement and 0 = no correlation.
correlation = spearmanr(predictions, y)[0]
print("Spearman's rank correlation:", correlation)

Spearman's rank correlation: 0.6441445125099904


In [13]:
import spacy
en = spacy.load('en_core_web_md')

##TODO compute word similarities in the WordSim353 dataset using spaCy word embeddings
##TODO compute spearman's rank correlation between these similarities and the human annotations
# Don't worry if results are not too convincing for this experiment

# Same evaluation as before, but using spaCy's pretrained word vectors instead
# of our own word2vec model.
spacy_predictions = []
for w1, w2 in X:
    # en(word) runs the word through spaCy; the resulting token carries a vector.
    tok1, tok2 = en(w1), en(w2)
    # has_vector is False for out-of-vocabulary words; if either word lacks a
    # vector we assign similarity 0 (consistent with how we handled our model).
    if tok1.has_vector and tok2.has_vector:
        # .similarity() computes the cosine similarity between the two vectors.
        spacy_predictions.append(tok1.similarity(tok2))
    else:
        spacy_predictions.append(0)

# Rank correlation between spaCy's similarities and the human annotations.
correlation = spearmanr(spacy_predictions, y)[0]
print("Spearman's rank correlation (spaCy):", correlation)

Spearman's rank correlation (spaCy): 0.4082453810278737


# Document Embeddings

**Task 1**
In this task, we evaluate different document embeddings on the English version of the [STS Benchmark](https://arxiv.org/pdf/1708.00055.pdf). The task is to determine how semantically similar two texts are and is a popular dataset to evaluate document embeddings, i.e. we want embeddings of two semantically similar documents to be similar as well. We provide a wordcounts baseline for this task and ask you to compute and evaluate embeddings for a selected sample of document embedding techniques.

To evaluate, we follow [(Reimers and Gurevych, 2019)](https://arxiv.org/pdf/1908.10084.pdf) and compute the Spearman’s rank correlation between the cosine-similarity of thesentence embeddings and the gold labels. 

In [14]:
# obtain the data (curl instead of wget for macOS compatibility)
!curl -L -o sts2017.eval.v1.1.zip http://alt.qcri.org/semeval2017/task1/data/uploads/sts2017.eval.v1.1.zip
!curl -L -o sts2017.gs.zip http://alt.qcri.org/semeval2017/task1/data/uploads/sts2017.gs.zip

!unzip -o sts2017.eval.v1.1.zip 
!unzip -o sts2017.gs.zip 

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   138  100   138    0     0    557      0 --:--:-- --:--:-- --:--:--   565
100 87902  100 87902    0     0  83566      0  0:00:01  0:00:01 --:--:--  243k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   138  100   138    0     0    526      0 --:--:-- --:--:-- --:--:--   530
100  3138  100  3138    0     0   4777      0 --:--:-- --:--:-- --:--:-- 12257
Archive:  sts2017.eval.v1.1.zip
  inflating: STS2017.eval.v1.1/LICENSE.txt  
  inflating: STS2017.eval.v1.1/README.txt  
  inflating: STS2017.eval.v1.1/STS.input.track1.ar-ar.txt  
  inflating: STS2017.eval.v1.1/STS.input.track2.ar-en.txt  
  inflating: STS2017.eval.v1.1/STS.input.track3.es-es.txt  
  inflating: STS2017.eval.v1.1/STS.input.track4a.es-en.txt  
  infl

In [15]:
# load the data

def load_STS_data():
    with open("STS2017.gs/STS.gs.track5.en-en.txt") as f:
        labels = [float(line.strip()) for line in f]
    
    text_a, text_b = [], []
    with open("STS2017.eval.v1.1/STS.input.track5.en-en.txt") as f:
        for line in f:
            line = line.strip().split("\t")
            text_a.append(line[0])
            text_b.append(line[1])
    return text_a, text_b, labels

text_a, text_b, labels = load_STS_data()
text_a[0], text_b[0], labels[0]

('A person is on a baseball team.',
 'A person is playing basketball on a team.',
 2.4)

In [16]:
# some utils
from scipy.stats import spearmanr
def evaluate(predictions, labels):
    print ("spearman's rank correlation", spearmanr(predictions, labels)[0])

import numpy as np
from numpy import dot
from numpy.linalg import norm

def cosine_similarity(a,b):
    return dot(a, b)/(norm(a)*norm(b))


In [17]:
# Wordcounts baseline
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer()
vec.fit(text_a + text_b)

# encode documents
text_a_encoded = np.array(vec.transform(text_a).todense())
text_b_encoded = np.array(vec.transform(text_b).todense())

# predict cosine similarities
predictions = [cosine_similarity(a,b) for a,b in zip(text_a_encoded, text_b_encoded)]

# evaluate
evaluate(predictions, labels)

spearman's rank correlation 0.6998056665685976


In [18]:
##TODO train Doc2Vec on the texts in the dataset
##TODO derive the word vectors for each text in the dataset
##TODO compute cosine similarity between the text pairs and evaluate spearman's rank correlation
## Don't worry if results are not satisfactory using Doc2Vec (the dataset is too small to train good embeddings)

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# Doc2Vec expects each training example as a TaggedDocument: a list of tokens
# plus a unique tag (here just the index). We train on all texts from both
# columns so the model sees the whole dataset.
documents = [TaggedDocument(words=doc.split(), tags=[i])
             for i, doc in enumerate(text_a + text_b)]

# Train the Doc2Vec model. vector_size sets the embedding dimensionality and
# min_count drops very rare words; the dataset is tiny so we keep min_count low.
doc2vec_model = Doc2Vec(documents, vector_size=100, min_count=2, epochs=40)

# infer_vector embeds a new (tokenized) document into the learned vector space.
# We embed every text in each pair, then take the cosine similarity per pair.
text_a_encoded = [doc2vec_model.infer_vector(doc.split()) for doc in text_a]
text_b_encoded = [doc2vec_model.infer_vector(doc.split()) for doc in text_b]

predictions = [cosine_similarity(a, b) for a, b in zip(text_a_encoded, text_b_encoded)]

# evaluate against the gold similarity labels (reuses the helper from above)
evaluate(predictions, labels)

spearman's rank correlation 0.284761349432108


In [19]:
##TODO do the same with embeddings provided by spaCy

# spaCy gives each document a single vector by averaging its token vectors,
# accessible via the .vector attribute of a processed Doc. We reuse the
# en_core_web_md model loaded earlier (it ships with word vectors).
text_a_encoded = [en(doc).vector for doc in text_a]
text_b_encoded = [en(doc).vector for doc in text_b]

# cosine similarity per text pair, then evaluate against the gold labels
predictions = [cosine_similarity(a, b) for a, b in zip(text_a_encoded, text_b_encoded)]
evaluate(predictions, labels)

spearman's rank correlation 0.5658556059546455


In [ ]:
##TODO do the same with universal sentence embeddings

# Google's Universal Sentence Encoder (USE) is served via TensorFlow Hub.
# Install once if needed:  !pip install tensorflow tensorflow_hub
import tensorflow_hub as hub

# Load the pretrained encoder (downloads & caches on first run). It maps a list
# of raw sentences directly to fixed-size 512-dim embeddings.
use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

# Encode all sentences in one batch each. The result is a tf tensor; numpy()
# converts it to a plain array we can feed to cosine_similarity.
text_a_encoded = use_model(text_a).numpy()
text_b_encoded = use_model(text_b).numpy()

# cosine similarity per text pair, then evaluate against the gold labels
predictions = [cosine_similarity(a, b) for a, b in zip(text_a_encoded, text_b_encoded)]
evaluate(predictions, labels)

/opt/anaconda3/envs/tad_courses/lib/python3.11/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


**Task 2**
Use your favorite document embeddings method to compute embeddings for a dataset you are interested in. Think of a method and provide some data visualization statistics (one method would be the path we have chosen in the notebook, i.e. cluster the embeddings with k-means and visualize low-dimensional representations of the document embeddings obtained by PCA). 

This task is very open and there is no right or wrong; If you want to use document embeddings in your course project, this is a chance to play around with them.



# Parsing

In [1]:
import pandas as pd
import nltk
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label) 
df["text"] = df["title"] + " " + df["lead"]
df = df.sample(n=10000) # # only use 10K datapoints
df.head()

,label,title,lead,text
105630,sci/tech,MSN Search Engine Uses Basis Technology for Na...,MSN Search Engine Uses Basis Technology for Na...,MSN Search Engine Uses Basis Technology for Na...
107389,sci/tech,Firms Push to Ease Tough Federal Scrutiny,Two-and-a-half years after Congress passed the...,Firms Push to Ease Tough Federal Scrutiny Two-...
76678,sci/tech,The iPod's New Trick: Photo Show,"A new, top-of-the-line iPod takes the concept ...","The iPod's New Trick: Photo Show A new, top-of..."
35896,business,FedEx Quarterly Earnings More Than Double,"NEW YORK (Reuters) - FedEx Corp. &lt;A HREF=""...",FedEx Quarterly Earnings More Than Double NEW...
119157,sci/tech,Linux can gain from the Firefox ad,Mozilla took out an ostensibly dual-purpose ad...,Linux can gain from the Firefox ad Mozilla too...


In [4]:
import spacy
nlp = spacy.load('en_core_web_sm')
#TODO preprocess the corpus using spacy or load the pre-processed corpus

# Run every text through spaCy so we get parsed Doc objects (tokens with
# part-of-speech tags, lemmas, and dependency relations) for the extraction
# tasks below. nlp.pipe processes texts in batches, which is much faster than
# calling nlp(text) in a loop. We disable the named-entity recognizer since we
# only need the dependency parse here.
docs = list(nlp.pipe(df["text"], disable=["ner"]))

# Keep the parsed docs alongside the dataframe so we can group them by label later.
df["doc"] = docs

print("processed", len(docs), "documents")

KeyboardInterrupt: 

### Information Extraction

In [3]:
def extract_subject_verb_pairs(sent):
    subjs = [w for w in sent if w.dep_ == "nsubj"]
    pairs = [(w.lemma_.lower(), w.head.lemma_.lower()) for w in subjs]
    return pairs

##TODO extract the subject-verbs pairs and print the result for the first document
# A nominal subject (dep_ == "nsubj") attaches to its governing verb via .head,
# so each pair is (subject_lemma, verb_lemma). A Doc can contain several
# sentences, so we collect pairs from every sentence (.sents) in the document.
first_doc = df["doc"].iloc[0]
print(first_doc.text)
print([pair for sent in first_doc.sents
            for pair in extract_subject_verb_pairs(sent)])

from collections import Counter
counter = Counter()

##TODO create a list ranking the most common pairs and print the first 10 items
# Tally the subject-verb pairs across every sentence of every document.
for doc in df["doc"]:
    for sent in doc.sents:
        counter.update(extract_subject_verb_pairs(sent))

# most_common(n) returns the n (pair, count) tuples sorted by descending count.
for pair, count in counter.most_common(10):
    print(pair, count)

KeyError: 'doc'

In [ ]:
##TODO do the same for verbs-object pairs ('dobj')
def extract_verb_object_pairs(sent):
    # A direct object (dep_ == "dobj") attaches to its governing verb via .head,
    # so we order each pair as (verb_lemma, object_lemma).
    objs = [w for w in sent if w.dep_ == "dobj"]
    pairs = [(w.head.lemma_.lower(), w.lemma_.lower()) for w in objs]
    return pairs

##TODO create a list ranking the most common pairs and print the first 10 items
counter = Counter()
for doc in df["doc"]:
    for sent in doc.sents:
        counter.update(extract_verb_object_pairs(sent))

for pair, count in counter.most_common(10):
    print(pair, count)

In [ ]:
##TODO do the same for adjectives-nouns pairs ('amod')
def extract_adjective_noun_pairs(sent):
    # An adjectival modifier (dep_ == "amod") attaches to the noun it modifies
    # via .head, so each pair is (adjective_lemma, noun_lemma).
    adjs = [w for w in sent if w.dep_ == "amod"]
    pairs = [(w.lemma_.lower(), w.head.lemma_.lower()) for w in adjs]
    return pairs

##TODO create a list ranking the most common pairs and print the first 10 items
counter = Counter()
for doc in df["doc"]:
    for sent in doc.sents:
        counter.update(extract_adjective_noun_pairs(sent))

for pair, count in counter.most_common(10):
    print(pair, count)

### Exploring cross label dependencies

In [5]:
##TODO extract all the subject-verbs and verbs-object pairs for the verb "win"
# Reuse the extractors from above and keep only the pairs whose verb is "win".
# In a subject-verb pair the verb is the 2nd element; in a verb-object pair it
# is the 1st element.
win_subject_verb = []
win_verb_object = []
for doc in df["doc"]:
    for sent in doc.sents:
        win_subject_verb += [(s, v) for s, v in extract_subject_verb_pairs(sent) if v == "win"]
        win_verb_object += [(v, o) for v, o in extract_verb_object_pairs(sent) if v == "win"]

print("subject-verb pairs for 'win':", Counter(win_subject_verb).most_common(10))
print("verb-object pairs for 'win':", Counter(win_verb_object).most_common(10))

KeyError: 'doc'

In [6]:
##TODO for each label create a list ranking the most common subject-verbs pairs and one for the most common verbs-object pairs
##TODO print the 10 most common pairs for each of the two lists for the labels "sport" and "business"
from collections import defaultdict

# One Counter per label for each pair type. We walk the docs together with their
# label so every pair is attributed to the category of the document it came from.
subject_verb_by_label = defaultdict(Counter)
verb_object_by_label = defaultdict(Counter)

for label, doc in zip(df["label"], df["doc"]):
    for sent in doc.sents:
        subject_verb_by_label[label].update(extract_subject_verb_pairs(sent))
        verb_object_by_label[label].update(extract_verb_object_pairs(sent))

for label in ["sport", "business"]:
    print(f"\n=== {label} ===")
    print("most common subject-verb pairs:")
    for pair, count in subject_verb_by_label[label].most_common(10):
        print("  ", pair, count)
    print("most common verb-object pairs:")
    for pair, count in verb_object_by_label[label].most_common(10):
        print("  ", pair, count)

KeyError: 'doc'